In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Create a Spark session
spark = SparkSession.builder.appName("UsedCarML").getOrCreate()


In [2]:
# Load the dataset into a DataFrame
df = spark.read.csv("C:/Users/snega/Downloads/cleaned_01.csv", header=True, inferSchema=True)

# Show the first few rows of the dataset
df.show()


+------------+------------+-------------+----------------+------+------+---------------+-------+-------+----+-----+---------+
|back_legroom|daysonmarket|front_legroom|fuel_tank_volume|length|height|maximum_seating|mileage|  price|year|width|wheelbase|
+------------+------------+-------------+----------------+------+------+---------------+-------+-------+----+-----+---------+
|        35.1|         522|         41.2|            12.7| 166.6|  66.5|            5.0|    7.0|23141.0|2019| 79.6|    101.2|
|        38.1|         207|         39.1|            17.7| 181.0|  68.0|            7.0|    8.0|46500.0|2020| 85.6|    107.9|
|        37.6|         196|         39.0|            23.5| 195.1|  73.0|            7.0|   11.0|67430.0|2020| 87.4|    115.0|
|        38.1|         137|         39.1|            17.7| 181.0|  68.0|            7.0|    7.0|48880.0|2020| 85.6|    107.9|
|        37.1|         242|         40.2|            16.6| 188.9|  66.3|            5.0|   12.0|66903.0|2020| 84.4|   

In [3]:
# Drop rows with missing values
df = df.na.drop()

# Feature columns
feature_columns = [
    "back_legroom", "daysonmarket", "front_legroom", "fuel_tank_volume",
    "length", "height", "maximum_seating", "mileage", "year", "width", "wheelbase"
]

# Assemble features into a vector
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
df = assembler.transform(df)

# Select relevant columns
df = df.select("features", "price")


In [45]:

# Split the data into training and testing sets (80% training, 20% testing)
(training_data, testing_data) = df.randomSplit([0.8, 0.2], seed=42)


In [46]:
# Create a Random Forest Regressor model
rf = RandomForestRegressor(featuresCol="features", labelCol="price", numTrees=1000)

# Train the model
model = rf.fit(training_data)


: 

In [36]:
# Make predictions on the testing set
predictions = model.transform(testing_data)
predictions.select("features", "price", "prediction").show()


+--------------------+-------+------------------+
|            features|  price|        prediction|
+--------------------+-------+------------------+
|[0.0,3.0,41.3,27....|10995.0|21244.204111791485|
|[0.0,5.0,41.4,26....|12995.0| 18531.83842091508|
|[0.0,6.0,41.3,26....| 8995.0| 12220.86291120089|
|[0.0,6.0,41.3,26....| 5499.0|12173.991755890358|
|[0.0,8.0,41.3,26....|10525.0| 14656.41755072836|
|[0.0,11.0,41.3,26...| 8488.0| 13344.77571834738|
|[0.0,12.0,41.3,34...| 9000.0|14035.494969386576|
|[0.0,13.0,41.3,26...| 8996.0|21765.863413089737|
|[0.0,14.0,41.3,26...|14980.0|16070.515153816808|
|[0.0,14.0,41.3,34...| 4990.0|12844.700445700555|
|[0.0,14.0,41.4,26...|10995.0| 18836.28436120117|
|[0.0,18.0,40.9,24...| 2640.0|12751.064536153172|
|[0.0,18.0,41.0,38...|19999.0| 16615.91930774411|
|[0.0,20.0,41.3,26...| 5999.0|12245.538943729645|
|[0.0,23.0,41.7,18...| 7800.0|8033.1962837549745|
|[0.0,27.0,41.0,38...| 7995.0| 15556.83521975941|
|[0.0,27.0,41.3,26...| 8900.0|13253.462009829338|


In [38]:
# Evaluate the model using Root Mean Squared Error (RMSE)
evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE): {rmse}")


Root Mean Squared Error (RMSE): 1.135209


In [39]:
#using linear regression
from pyspark.ml.regression import LinearRegression
lr = LinearRegression(featuresCol="features", labelCol="price")

# Train the model
model_lr = lr.fit(training_data)

predictions_lr = model_lr.transform(testing_data)
predictions_lr.select("features", "price", "prediction")


+--------------------+-------+------------------+
|            features|  price|        prediction|
+--------------------+-------+------------------+
|[0.0,3.0,41.3,27....|10995.0| 24391.79991886625|
|[0.0,5.0,41.4,26....|12995.0| 29636.37092604139|
|[0.0,6.0,41.3,26....| 8995.0|12329.530886704335|
|[0.0,6.0,41.3,26....| 5499.0|11231.866019903682|
|[0.0,8.0,41.3,26....|10525.0| 22639.76461130893|
|[0.0,11.0,41.3,26...| 8488.0|12546.350086219143|
|[0.0,12.0,41.3,34...| 9000.0| 21706.96587787429|
|[0.0,13.0,41.3,26...| 8996.0| 25750.23450432229|
|[0.0,14.0,41.3,26...|14980.0| 25395.04837359884|
|[0.0,14.0,41.3,34...| 4990.0|20289.156438785605|
|[0.0,14.0,41.4,26...|10995.0| 29494.57228891831|
|[0.0,18.0,40.9,24...| 2640.0| 184.3052701742854|
|[0.0,18.0,41.0,38...|19999.0| 42107.11623538635|
|[0.0,20.0,41.3,26...| 5999.0|14408.564137894195|
|[0.0,23.0,41.7,18...| 7800.0|-2801.074627999915|
|[0.0,27.0,41.0,38...| 7995.0| 30968.47766674403|
|[0.0,27.0,41.3,26...| 8900.0|15304.228173548356|


In [41]:
# Evaluate the model using Root Mean Squared Error (RMSE)
evaluator_lr = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="rmse")
rmse_lr = evaluator_lr.evaluate(predictions_lr)
print(f"Root Mean Squared Error (RMSE): {rmse_lr}")

Root Mean Squared Error (RMSE): 4.89021891


In [42]:
from pyspark.ml.regression import GBTRegressor
gb = GBTRegressor(featuresCol="features", labelCol="price")

# Train the model
model_gb = gb.fit(training_data)

predictions_gb = model_gb.transform(testing_data)
predictions_gb.select("features", "price", "prediction")


+--------------------+-------+------------------+
|            features|  price|        prediction|
+--------------------+-------+------------------+
|[0.0,3.0,41.3,27....|10995.0|12138.424950978368|
|[0.0,5.0,41.4,26....|12995.0| 18206.86761400346|
|[0.0,6.0,41.3,26....| 8995.0| 9494.326122482233|
|[0.0,6.0,41.3,26....| 5499.0| 7562.552432734713|
|[0.0,8.0,41.3,26....|10525.0|14229.317480604877|
|[0.0,11.0,41.3,26...| 8488.0|12754.293503987637|
|[0.0,12.0,41.3,34...| 9000.0|15398.392332483772|
|[0.0,13.0,41.3,26...| 8996.0|14229.317480604877|
|[0.0,14.0,41.3,26...|14980.0|14229.317480604877|
|[0.0,14.0,41.3,34...| 4990.0|10206.651261230847|
|[0.0,14.0,41.4,26...|10995.0| 18206.86761400346|
|[0.0,18.0,40.9,24...| 2640.0|  8792.91360323886|
|[0.0,18.0,41.0,38...|19999.0| 17795.22210608409|
|[0.0,20.0,41.3,26...| 5999.0|12138.424950978368|
|[0.0,23.0,41.7,18...| 7800.0|6973.0311981622135|
|[0.0,27.0,41.0,38...| 7995.0|16320.198129466848|
|[0.0,27.0,41.3,26...| 8900.0|16230.290001747999|


In [44]:
# Evaluate the model using Root Mean Squared Error (RMSE)
evaluator_gb = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="rmse")
rmse_gb = evaluator_gb.evaluate(predictions_gb)
print(f"Root Mean Squared Error (RMSE): {rmse_gb}")

Root Mean Squared Error (RMSE): 28.4414333429576
